# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [86]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets
import sys
sys.path.append('../05_src/')

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [87]:
import pypdf
from langchain_core.documents import Document

def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

def get_document_content(path):
    docs = load_pdf_pages(path)
    document_text = ""
    for page in docs:
        document_text += page.page_content + "\n"
    return document_text

In [88]:
file_path = "../05_src/documents/ai_report_2025.pdf"
content = get_document_content(file_path)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [89]:
from utils.clients import get_client
from pydantic import BaseModel
import os
from typing import List, Tuple, Optional
from openai import OpenAI
from IPython.display import display, Markdown

os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

In [90]:
class DocumentSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [137]:
INSTRUCTIONS = """
You are a helpful assistant that summarizes professional AI-related articles.

You produce accurate, structured summaries for AI professionals.
You are careful, factual, and concise.
You do not invent information.
You preserve important quantitative claims, methods, limitations, and conclusions from the article.

The summary must be written in the following distinguishable tone:
{tone}

Return only the requested structured output.
"""

PROMPT = """
Summarize the following article for an AI professional.
{text}

Requirements:
- Identify the article title and author if available.
- Write a Relevance statement explaining why this article matters for an AI professional's development.
- Write a concise Summary no longer than 1000 tokens.
- Use the requested tone: {tone}.
- Preserve key findings, quantitative claims, methodology, limitations, and conclusions.
- Do not fabricate unsupported claims.
- Do not include token counts in the text; token counts will be filled programmatically from the response object.
"""

In [138]:
class DocumentSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int | None = None
    OutputTokens: int | None = None

In [139]:
def get_article_summary(
    text: str,
    tone: str = "Legalese",
    model: str = "gpt-4o-mini",
    max_tokens: int = 1000,
    temperature: float = 0.2,
):
    response = client.responses.parse(
        model=model,
        instructions=INSTRUCTIONS.format(tone=tone),
        input=[
            {
                "role": "system",
                "content": PROMPT.format(tone=tone, text=text),
            },
            {
                "role": "user",
                "content": text,
            },
        ],
        text_format=DocumentSummary,
        max_output_tokens=max_tokens,
        temperature=temperature,
    )

    parsed = response.output_parsed

    # attach token usage
    parsed.InputTokens = response.usage.input_tokens
    parsed.OutputTokens = response.usage.output_tokens

    return parsed

In [140]:
summary = get_article_summary(text=content, tone='Legalese')

In [125]:
import textwrap

def print_summary(summary, width=100):
    def wrap(text):
        return textwrap.fill(str(text), width=width)

    print(f"Author: {summary.Author}\n")
    print(f"Title: {summary.Title}\n")
    print(f"Relevance:\n{wrap(summary.Relevance)}\n")
    print(f"Summary:\n{wrap(summary.Summary)}\n")
    print(f"Tone: {summary.Tone}\n")
    print(f"Number of input tokens: {summary.InputTokens}")
    print(f"Number of output tokens: {summary.OutputTokens}")


In [141]:
print_summary(summary)

Author: MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

Title: The GenAI Divide: State of AI in Business 2025

Relevance:
This article provides critical insights into the current state of generative AI (GenAI) adoption in
enterprises, highlighting the significant gap between high adoption rates and low transformational
impact. Understanding these dynamics is essential for AI professionals aiming to navigate the
complexities of AI implementation and maximize ROI in their organizations.

Summary:
The report reveals a stark disparity in the outcomes of generative AI (GenAI) initiatives, termed
the 'GenAI Divide,' where 95% of organizations report no measurable return on a collective
investment of $30–40 billion. Despite widespread adoption of tools like ChatGPT, only 5% of
integrated AI pilots yield significant value, primarily due to issues such as brittle workflows and
lack of contextual learning. The research, based on over 300 AI initiatives and interviews 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [142]:
input = content
actual_output = summary.Summary

In [ ]:
summarization_questions = [
    "Does the summary accurately explain the concept of the 'GenAI Divide' as the gap between widespread AI adoption and limited measurable business transformation?",
    "Does the summary correctly report the report’s central quantitative findings, including high enterprise GenAI investment, widespread pilot activity, and the low rate of successful production deployment?",
    "Does the summary distinguish between general-purpose AI tools (e.g., ChatGPT, Copilot) and custom enterprise AI systems in terms of usability, deployment success, and business impact?",
    "Does the summary explain the primary reasons enterprise GenAI pilots fail, including poor workflow integration, lack of memory, limited contextual adaptation, and weak learning capability?",
    "Does the summary accurately convey the report’s argument that the core barrier to enterprise AI success is organizational and architectural learning failure rather than model quality or regulation?",
    "Does the summary capture the report’s discussion of the 'shadow AI economy,' including employees independently using consumer AI tools more successfully than official enterprise systems?",
    "Does the summary reflect the report’s findings regarding AI investment allocation, particularly the concentration of spending in sales and marketing despite stronger ROI potential in back-office operations?",
    "Does the summary describe the characteristics of organizations and vendors that successfully cross the GenAI Divide, including workflow customization, learning capability, adaptability, and strategic partnerships?",
    "Does the summary preserve the report’s nuanced discussion of workforce impact, emphasizing selective displacement and operational efficiency rather than broad layoffs?",
    "Does the summary explain the report’s vision of agentic AI and the emerging 'Agentic Web,' including persistent memory, interoperable agents, adaptive workflows, and autonomous coordination?"
]

coherence_questions = [
    "Does the summary present ideas in a logical and easy-to-follow progression?",
    "Does the summary clearly distinguish between consumer AI tools and enterprise AI systems without introducing ambiguity or contradiction?",
    "Does the summary coherently connect the report’s major themes, including pilot failure, workflow integration challenges, learning capability, and organizational adoption barriers?",
    "Does the summary use terminology consistently when discussing concepts such as the GenAI Divide, learning systems, workflow adaptation, and agentic AI?",
    "Does the summary maintain smooth transitions between topics such as adoption trends, deployment outcomes, workforce impact, ROI, and future AI infrastructure?",
    "Does the summary avoid redundancy, fragmentation, or abrupt topic shifts that reduce readability?"
]

tonality_questions = [
    "Does the summary consistently adopt the tone explicitly requested in the prompt?",
    "Does the writing style reflect the intended tone without drifting into an unrelated or inconsistent voice?",
    "Does the chosen tone preserve the factual meaning and analytical intent of the original report rather than distorting or exaggerating it?",
    "Does the summary avoid tonal mismatches such as excessive hype, sarcasm, informality, or emotional language when they are not requested?",
    "Does the summary maintain a coherent tone throughout the entire response rather than switching styles between sections?",
    "Does the tone remain appropriate for summarizing a research and business-oriented report, even when stylistic modifications are requested?",
    "Does the summary balance stylistic tone adherence with clarity, professionalism, and readability?"
]

safety_questions = [
    "Does the summary avoid fabricating statistics, deployment rates, ROI claims, or findings that are not supported by the report?",
    "Does the summary avoid overstating speculative claims or presenting future projections as established facts?",
    "Does the summary preserve uncertainty, caveats, and methodological limitations discussed in the report?",
    "Does the summary avoid exaggerating workforce displacement, automation impact, or organizational transformation beyond what the report claims?",
    "Does the summary avoid attributing unsupported endorsements, guarantees, or conclusions to organizations, technologies, or researchers?",
    "Does the summary accurately represent correlations, observations, and interview findings without implying stronger causal conclusions than the report supports?"
]

In [145]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCaseParams


def format_questions(questions):
    return "\n".join([f"{i+1}. {q}" for i, q in enumerate(questions)])


def evaluate_summary(
    context: str,
    actual_output: str,
    tone: str,
    model: str,
    summarization_questions: list[str],
    coherence_questions: list[str],
    tonality_questions: list[str],
    safety_questions: list[str],
):
    test_case = LLMTestCase(
        input=context,
        actual_output=actual_output
    )

    summarization_metric = SummarizationMetric(
        model=model,
        assessment_questions=summarization_questions
    )

    coherence_metric = GEval(
        name="Coherence",
        criteria=f"""
        Evaluate whether the summary is coherent, logically structured, and easy to follow.

        The summary should organize the article's main ideas in a logical sequence,
        avoid confusing transitions, and clearly explain relationships between ideas.

        Requested tone:
        {tone}

        Assessment questions:
        {format_questions(coherence_questions)}
        """,
        evaluation_steps=[
            "Check whether the summary has a clear beginning, middle, and end.",
            "Check whether the ideas are presented in a logical order.",
            "Check whether relationships between claims are clearly explained.",
            "Penalize vague, confusing, repetitive, or poorly connected writing.",
            "Penalize omissions that make the article's argument difficult to understand."
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT
        ]
    )

    tonality_metric = GEval(
        name="Tonality",
        criteria=f"""
        Evaluate whether the summary adequately adopts the explicitly requested tone.

        The requested tone is:
        {tone}

        The summary should use a recognizable and consistent version of that tone
        while remaining accurate, readable, and appropriate for the article.

        Assessment questions:
        {format_questions(tonality_questions)}
        """,
        evaluation_steps=[
            f"Check whether the actual output uses the requested tone: {tone}.",
            "Check whether the tone is consistent throughout the summary.",
            "Check whether the tone is distinguishable rather than generic.",
            "Penalize tone that interferes with factual clarity or readability.",
            "Penalize summaries that ignore the requested tone or switch tones unexpectedly."
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT
        ]
    )

    safety_metric = GEval(
        name="Safety",
        criteria=f"""
        Evaluate whether the summary is safe, factual, and not misleading.

        The summary should accurately reflect the source article and avoid unsupported
        claims, fabricated statistics, exaggerated conclusions, or misleading certainty.

        Assessment questions:
        {format_questions(safety_questions)}
        """,
        evaluation_steps=[
            "Check whether facts in the actual output contradict any facts in the input.",
            "Check whether the actual output invents statistics, findings, authors, or conclusions not supported by the input.",
            "Penalize omission of important details when the omission makes the summary misleading.",
            "Penalize vague language that obscures the article's actual claims.",
            "Penalize contradictions of the article's stated opinions, arguments, limitations, or conclusions."
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT
        ]
    )

    metrics = [
        summarization_metric,
        coherence_metric,
        tonality_metric,
        safety_metric
    ]

    for metric in metrics:
        metric.measure(test_case)

    results = {
        "Summarization": {
            "score": summarization_metric.score,
            "reason": summarization_metric.reason
        },
        "Coherence": {
            "score": coherence_metric.score,
            "reason": coherence_metric.reason
        },
        "Tonality": {
            "score": tonality_metric.score,
            "reason": tonality_metric.reason
        },
        "Safety": {
            "score": safety_metric.score,
            "reason": safety_metric.reason
        }
    }

    return results, metrics

/var/folders/36/lk67tbn1567_fc3pz503h6z00000gq/T/ipykernel_19731/1803197901.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [149]:
results, metrics = evaluate_summary(
    context=content,
    actual_output=summary.Summary,
    tone="Legalese",
    model=MODEL,
    summarization_questions=summarization_questions,
    coherence_questions=coherence_questions,
    tonality_questions=tonality_questions,
    safety_questions=safety_questions
)

Output()

Output()

Output()

Output()

In [156]:
import textwrap

def print_evaluation(results, width=100):

    def wrap(text):
        return textwrap.fill(str(text), width=width)

    for metric_name, metric_data in results.items():

        score = metric_data.get("score", "N/A")
        reason = metric_data.get("reason", "No reason provided.")

        print(f"{metric_name} Score: {score:.3f}\n")

        print(f"{metric_name} Reason:")
        print(wrap(reason))
        print("\n" + "=" * width + "\n")

In [158]:
print_evaluation(results)

Summarization Score: 0.500

Summarization Reason:
The score is 0.50 because the summary contains significant contradictions to the original text,
particularly regarding the reported return on investment, and introduces extra information that was
not present in the original text, which may mislead the reader. Additionally, the summary fails to
address several key questions that the original text could answer, indicating a lack of depth and
fidelity to the source material.


Coherence Score: 0.800

Coherence Reason:
The summary has a clear arc: it introduces the GenAI Divide and core findings, explains major
supporting patterns and causes in the middle, and ends with the report’s recommended path forward.
Ideas are mostly in logical order, moving from headline statistics ($30–40B invested, 95% seeing no
return, only 5% of pilots creating value) to causes like the learning gap and then to implications
such as shadow AI and partnership-based success. It explains several relationships betwe

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [161]:
import json

REFINEMENT_INSTRUCTIONS = """
You are an expert report summarizer and editor.

You revise summaries using source material and evaluation feedback.
You must preserve factual accuracy, avoid unsupported claims, and improve only what the evaluation identifies as weak.

Do not invent facts, statistics, authors, conclusions, or citations.
Do not contradict the original document.
Do not add information that is not supported by the original document.
Return only the improved summary text.
"""

REFINEMENT_PROMPT = """
You are revising a document summary based on evaluation feedback.

Original document:
<context>
{content}
</context>

Previous summary:
<previous_summary>
{summary}
</previous_summary>

Evaluation feedback:
<evaluation_feedback>
{results}
</evaluation_feedback>

Requested tone:
{tone}

Your goals are to:
- Address weaknesses identified in the evaluation feedback
- Preserve factual accuracy
- Correct any contradictions with the original document
- Add important omitted details if they are necessary for fidelity
- Improve organization and readability
- Use the requested tone consistently: {tone}
- Avoid hallucinations or unsupported claims
- Keep the summary concise and no longer than 1000 tokens

Generate an improved summary.
"""

In [163]:
def improve_summary(
    content: str,
    summary: str,
    results: dict,
    tone: str,
    model: str = MODEL,
    max_tokens: int = 1000,
    temperature: float = 0.2,
):
    refinement_prompt = REFINEMENT_PROMPT.format(
        content=content,
        summary=summary,
        results=json.dumps(results, indent=2),
        tone=tone
    )

    improved_completion = client.responses.create(
        model=model,
        temperature=temperature,
        max_output_tokens=max_tokens,
        input=[
            {
                "role": "developer",
                "content": REFINEMENT_INSTRUCTIONS
            },
            {
                "role": "user",
                "content": refinement_prompt
            }
        ]
    )

    return improved_completion.output_text, improved_completion

In [164]:
improved_summary, improved_completion = improve_summary(
    content=content,
    summary=summary.Summary,
    results=results,
    tone="Legalese",
    model=MODEL
)

In [165]:
print(textwrap.fill(str(improved_summary), width=100))

The report delineates a pronounced disparity in the outcomes of generative AI (GenAI) initiatives,
referred to as the 'GenAI Divide,' wherein 95% of organizations report no measurable return on a
collective investment estimated between $30 billion and $40 billion. Despite the extensive adoption
of tools such as ChatGPT, only 5% of integrated AI pilots yield significant value, primarily
attributable to challenges including brittle workflows and insufficient contextual learning. The
research methodology encompasses a systematic review of over 300 publicly disclosed AI initiatives,
structured interviews with representatives from 52 organizations, and survey responses from 153
senior leaders across four major industry conferences.  The report identifies four salient patterns
contributing to the GenAI Divide: (1) limited disruption across sectors, with only two of eight
major sectors exhibiting meaningful structural change; (2) an enterprise paradox wherein large firms
lead in pilot volume 

In [168]:
test_case = LLMTestCase(
    input=content,
    actual_output=summary.Summary)

improved_test_case = LLMTestCase(
    input=content,
    actual_output=improved_summary
)

comparison = {
    'Original': {},
    'Improved': {}
}
for metric in metrics:
    comparison['Original'][metric] = metric.measure(test_case)
    comparison['Improved'][metric] = metric.measure(improved_test_case)

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

In [169]:
def print_comparison(results):
    metric_names = {
        0: "Summarization",
        1: "Coherence",
        2: "Tonality",
        3: "Safety",
    }

    original_items = list(results["Original"].items())
    improved_items = list(results["Improved"].items())

    print("=" * 80)
    print(f"{'Metric':<20} {'Original':<12} {'Improved':<12} {'Delta':<12}")
    print("=" * 80)

    for idx, ((_, orig_score), (_, imp_score)) in enumerate(zip(original_items, improved_items)):
        delta = imp_score - orig_score
        metric_name = metric_names.get(idx, f"Metric_{idx}")

        print(
            f"{metric_name:<20} "
            f"{orig_score:<12.3f} "
            f"{imp_score:<12.3f} "
            f"{delta:+.3f}"
        )

    print("=" * 80)

In [171]:
print_comparison(comparison)

Metric               Original     Improved     Delta       
Summarization        0.438        0.529        +0.092
Coherence            0.800        0.800        +0.000
Tonality             0.138        0.795        +0.658
Safety               0.888        0.806        -0.082


The refinement process improved the overall quality of the generated summary, particularly in terms of summarization fidelity and tone adherence.

The Summarization score increased from 0.438 to 0.529 (+0.092), suggesting that the revised prompt successfully incorporated more relevant details from the source article and reduced some of the contradictions or omissions identified during the initial evaluation.

The Tonality score increased significantly from 0.138 to 0.795 (+0.658), which indicates that the refinement prompt was highly effective at enforcing consistent adoption of the requested writing style.

The Coherence score remained stable at 0.800, suggesting that the original summary was already logically structured and easy to follow. The refinement process simply preserved this organizational quality without introducing additional confusion or fragmentation.

Lastly, the Safety score decreased slightly from 0.888 to 0.806 (-0.082). This suggests that optimizing for one dimension (such as tone or completeness) can unintentionally reduce performance in another dimension (such as factual conservativeness or safety).

Overall, the refinement loop demonstrated that evaluation-guided prompting can meaningfully improve summary quality. Nevertheless, tradeoff are an inherent limitation of iterative self-correction systems.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
